# 02 DAG and Discovery Review

Review-only notebook for the hand-built DAG and saved reduced causal discovery outputs. This notebook does not run causal discovery; use `make run-discovery` to refresh artifacts.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import yaml
from IPython.display import Image, Markdown, display

from oulad_causal.config import DOCS_DIR, FIGURES_DIR, PROCESSED_DATA_DIR
from oulad_causal.dag import (
    DAG_FIGURE_PATH,
    DAG_VARIABLE_AVAILABILITY_PATH,
    DAG_YAML_PATH,
    discovery_variable_list,
    primary_dag_spec,
    recommended_baseline_adjustment_set,
)
from oulad_causal.discovery import (
    DISCOVERY_COMBINED_EDGES_PATH,
    DISCOVERY_COMPARISON_PATH,
    DISCOVERY_METADATA_PATH,
    DISCOVERY_PREPROCESSING_MAP_PATH,
    DISCOVERY_STABILITY_PATH,
    DISCOVERY_SUMMARY_PATH,
    select_discovery_variables,
)

## Hand-Built DAG

The saved DAG is the domain-informed baseline for identification. Discovery output is compared against this graph rather than treated as definitive truth.

In [ ]:
if DAG_YAML_PATH.exists():
    with DAG_YAML_PATH.open(encoding="utf-8") as handle:
        dag_spec = yaml.safe_load(handle)
else:
    dag_spec = primary_dag_spec()

pd.DataFrame(dag_spec["nodes"])[["id", "label", "role", "observed", "cohort_columns"]]

In [ ]:
if DAG_FIGURE_PATH.exists():
    display(Image(filename=str(DAG_FIGURE_PATH)))
else:
    print(f"DAG figure not found yet: {DAG_FIGURE_PATH}")

## Primary Adjustment Set

In [ ]:
pd.Series(recommended_baseline_adjustment_set(), name="recommended_baseline_adjustment_column").to_frame()

## Cohort Metadata and Variable Availability

In [ ]:
summary_path = PROCESSED_DATA_DIR / "cohort_summary.json"
with summary_path.open(encoding="utf-8") as handle:
    cohort_summary = json.load(handle)

cohort_summary["cohort_size"], cohort_summary["primary_window_days"], cohort_summary["exclusion_counts"]

In [ ]:
if DAG_VARIABLE_AVAILABILITY_PATH.exists():
    display(pd.read_csv(DAG_VARIABLE_AVAILABILITY_PATH))
else:
    print(f"Saved availability table not found yet: {DAG_VARIABLE_AVAILABILITY_PATH}")

## Discovery Variable Set

The implemented workflow uses a bounded 12-variable set for PC, FCI, and GES. It excludes later assessment behavior and the continuous engagement z-score to avoid a deterministic score-threshold relationship dominating discovery.

In [ ]:
pd.Series(select_discovery_variables(), name="implemented_discovery_variable").to_frame()

## Discovery Run Metadata

In [ ]:
if DISCOVERY_METADATA_PATH.exists():
    with DISCOVERY_METADATA_PATH.open(encoding="utf-8") as handle:
        discovery_metadata = json.load(handle)
    display(pd.DataFrame(discovery_metadata["methods"]).T)
    display(discovery_metadata.get("artifacts", {}))
else:
    print(f"Discovery metadata not found yet: {DISCOVERY_METADATA_PATH}")

## Discovery Preprocessing

In [ ]:
if DISCOVERY_PREPROCESSING_MAP_PATH.exists():
    with DISCOVERY_PREPROCESSING_MAP_PATH.open(encoding="utf-8") as handle:
        preprocessing = json.load(handle)
    display(pd.DataFrame(preprocessing["columns"]).T)
else:
    print(f"Preprocessing map not found yet: {DISCOVERY_PREPROCESSING_MAP_PATH}")

## Discovery Edge Lists

In [ ]:
if DISCOVERY_COMBINED_EDGES_PATH.exists():
    edges = pd.read_csv(DISCOVERY_COMBINED_EDGES_PATH)
    display(edges)
    display(edges.groupby("method").size().rename("edge_count").to_frame())
else:
    print(f"Combined discovery edge list not found yet: {DISCOVERY_COMBINED_EDGES_PATH}")

## Stability Checks

In [ ]:
if DISCOVERY_STABILITY_PATH.exists():
    stability = pd.read_csv(DISCOVERY_STABILITY_PATH)
    display(stability.sort_values(["edge_frequency", "method"], ascending=[False, True]).head(30))
else:
    print(f"Discovery stability table not found yet: {DISCOVERY_STABILITY_PATH}")

## Hand-DAG Comparison

In [ ]:
if DISCOVERY_COMPARISON_PATH.exists():
    comparison = pd.read_csv(DISCOVERY_COMPARISON_PATH)
    display(comparison)
    if not comparison.empty:
        display(comparison.groupby(["method", "in_hand_skeleton", "in_hand_directed"]).size().rename("count").reset_index())
else:
    print(f"Discovery comparison table not found yet: {DISCOVERY_COMPARISON_PATH}")

## Discovery Graph Figures

In [ ]:
for method in ["pc", "fci", "ges"]:
    figure_path = FIGURES_DIR / f"discovery_{method}.png"
    if figure_path.exists():
        print(method.upper())
        display(Image(filename=str(figure_path)))
    else:
        print(f"{method.upper()} figure not found: {figure_path}")

## Generated Discovery Summary

In [ ]:
if DISCOVERY_SUMMARY_PATH.exists():
    display(Markdown(DISCOVERY_SUMMARY_PATH.read_text(encoding="utf-8")))
else:
    print(f"Discovery summary not found yet: {DISCOVERY_SUMMARY_PATH}")